In [1]:
from polyscanner.env import load_env
import os, psycopg
import pandas as pd

load_env()
DB_URL = os.getenv("DATABASE_URL").replace("postgresql+psycopg://", "postgresql://")
conn = psycopg.connect(DB_URL)


In [2]:
with conn.cursor() as cur:
    cur.execute("select count(*) from pm_market;")
    n = cur.fetchone()[0]
n


200

In [3]:
terms = [
    "fed","fomc","powell","cpi","inflation","rate cut","rate hike","basis points","yield","treasury",
    "sec","cftc","etf","stablecoin","crypto regulation",
    "tariff","export control","sanction","asml","tsmc","nvidia","china","taiwan","invasion","blockade",
    "antitrust","doj","ftc","app store",
    "medicare","medicaid","cms","reimbursement",
    "delinquency","credit card","default","mortgage",
]

rows = []
with conn.cursor() as cur:
    for t in terms:
        cur.execute(
            "select count(*) from pm_market where lower(question) like %s;",
            (f"%{t.lower()}%",),
        )
        rows.append((t, cur.fetchone()[0]))

df_counts = pd.DataFrame(rows, columns=["term","count"]).sort_values("count", ascending=False)
df_counts.head(25)


,term,count
0,fed,6
21,china,1
22,taiwan,1
15,tariff,1
27,ftc,0
23,invasion,0
24,blockade,0
25,antitrust,0
26,doj,0
28,app store,0


In [4]:
df_top_vol = pd.read_sql_query(
    """
    select pm_market_id, volume_usd, question
    from pm_market
    order by volume_usd desc nulls last
    limit 30;
    """,
    conn,
)
df_top_vol

/var/folders/_h/pwtmbd9d6nxcz50zwwzs0w6w0000gn/T/ipykernel_7277/394032048.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_vol = pd.read_sql_query(


,pm_market_id,volume_usd,question
0,559684,3.757616e+07,Will Chelsea Clinton win the 2028 Democratic p...
1,553861,3.244421e+07,Will the Indiana Pacers win the 2026 NBA Finals?
2,553874,3.121312e+07,Will the Memphis Grizzlies win the 2026 NBA Fi...
3,559683,3.062389e+07,Will George Clooney win the 2028 Democratic pr...
4,559671,2.962615e+07,Will Zohran Mamdani win the 2028 Democratic pr...
5,559677,2.901285e+07,Will Hillary Clinton win the 2028 Democratic p...
6,559679,2.897348e+07,Will Bernie Sanders win the 2028 Democratic pr...
7,559681,2.824250e+07,Will LeBron James win the 2028 Democratic pres...
8,559680,2.748531e+07,Will Phil Murphy win the 2028 Democratic presi...
9,559678,2.687947e+07,Will Liz Cheney win the 2028 Democratic presid...
